# School Choice Through Three Research Personas

## From strategic thinking to social choice to mechanism design

**COMSCI/ECON 206 · Computational Microeconomics · Prof. Luyao Zhang**

This notebook turns one school-choice problem into three disciplined research conversations:

1. **Game theorist:** Who are the players? What are their strategy sets, strategy profiles, information, and payoffs? Can someone gain by changing only their report?
2. **Social-choice researcher:** Which collective goals—access, welfare, fairness, efficiency, priority respect, or legitimacy—should the assignment pursue, and where do those goals conflict?
3. **Mechanism designer:** Which allocation rule can implement those goals when participants respond strategically? Does it satisfy feasibility, stability, or strategy-proofness under stated assumptions?

> **Observable evidence:** explain the same result in all three personas, present mechanism pseudocode, execute both algorithms, inspect the animation, and change at least one input. A favorable simulation is not a general proof.

## Intellectual lineage: what each advance made visible

- **Strategic interaction.** Nash formalized mutual best responses; Harsanyi represented incomplete information using types and priors; Selten refined Nash by requiring credible behavior after every relevant history. See the [1994 Prize summary](https://www.nobelprize.org/prizes/economic-sciences/1994/summary/), [Nash (1950)](https://doi.org/10.1073/pnas.36.1.48), [Harsanyi (1967)](https://doi.org/10.1287/mnsc.14.3.159), and [Selten (1965)](https://www.jstor.org/stable/40748884).
- **Social choice.** Arrow's work clarified how individual preferences and collective criteria can conflict. See the [1972 Prize summary](https://www.nobelprize.org/prizes/economic-sciences/1972/summary/).
- **Mechanism design.** Hurwicz, Maskin, and Myerson studied which institutions implement social objectives when information and incentives constrain behavior. See the [2007 Prize summary](https://www.nobelprize.org/prizes/economic-sciences/2007/summary/).
- **Matching and market design.** Gale and Shapley's deferred-acceptance idea became a foundation for stable matching; Roth and Shapley were recognized for stable allocations and market design. See [Gale and Shapley (1962)](https://doi.org/10.2307/2312726) and the [2012 Prize summary](https://www.nobelprize.org/prizes/economic-sciences/2012/summary/).
- **School-choice institutions.** The mechanism-design approach and Boston evidence are developed by [Abdulkadiroğlu and Sönmez (2003)](https://doi.org/10.1257/000282803322157061) and [Abdulkadiroğlu et al. (2005)](https://doi.org/10.1257/000282805774669637).

The sequence is conceptual rather than a claim that one field replaced another: game theory analyzes strategic interdependence; social choice articulates collective criteria; mechanism design asks which rules implement those criteria under information and incentive constraints.

## The minimum viable matching game

**Players.** Four students and three schools. If schools' priorities are policy inputs rather than strategic choices, the strategic players in this classroom model are the students. A richer model could make schools strategic too.

**Student strategy set.** Each student can submit multiple possible rank-order lists. There are therefore at least two players and at least two strategies per player; the displayed truthful list is one strategy, not the entire strategy set.

**Outcome and payoff.** A mechanism maps the submitted profile to an assignment. We use ordinal preferences: receiving a higher-ranked school is better. This is sufficient for the manipulation counterexample below but does not measure interpersonal welfare.

**Information.** Preferences and priorities are visible in this small teaching example. A research application must state which of them are known, private, estimated, or elicited.

| Student | 1st choice | 2nd choice | 3rd choice |
|---|---|---|---|
| Amina | Beacon | Aurora | Cedar |
| Bo | Beacon | Aurora | Cedar |
| Chen | Aurora | Beacon | Cedar |
| Dara | Aurora | Cedar | Beacon |

Every school uses priority **Amina ≻ Bo ≻ Chen ≻ Dara**. Capacities are Aurora 1, Beacon 1, and Cedar 2. This deliberately small example is a counterexample laboratory, not a model of any particular school district.

## Mechanism pseudocode

### Boston mechanism / immediate acceptance

```text
initialize every student as unmatched and every seat as open
for rank k = 1, 2, ...:
    each unmatched student applies to the school ranked k
    each school permanently accepts its highest-priority applicants
        up to remaining capacity
    reject all other applicants
stop after every rank has been considered
```

### Gale–Shapley student-proposing deferred acceptance

```text
initialize every student as unmatched and every school hold as empty
while an unmatched student has an untried school:
    each such student proposes to the next school on the submitted list
    each school considers its previous holds plus new proposals
    each school tentatively holds its highest-priority students
        up to capacity and rejects the rest
make all holds final when no further proposal is possible
```

The critical algorithmic difference is **final acceptance now** versus **tentative holding with possible displacement**.

In [1]:
from copy import deepcopy
from html import escape
from IPython.display import HTML, display
import pandas as pd

STUDENTS = ["Amina", "Bo", "Chen", "Dara"]
SCHOOLS = ["Aurora", "Beacon", "Cedar"]
CAPACITIES = {"Aurora": 1, "Beacon": 1, "Cedar": 2}
PREFERENCES = {
    "Amina": ["Beacon", "Aurora", "Cedar"],
    "Bo": ["Beacon", "Aurora", "Cedar"],
    "Chen": ["Aurora", "Beacon", "Cedar"],
    "Dara": ["Aurora", "Cedar", "Beacon"],
}
PRIORITIES = {school: STUDENTS.copy() for school in SCHOOLS}

In [2]:
def priority_rank(priorities, school, student):
    """Smaller values mean higher school priority."""
    return priorities[school].index(student)

def snapshot(round_number, label, by_student, by_school, applications=None):
    return {
        "round": round_number,
        "label": label,
        "applications": deepcopy(applications or {}),
        "by_student": deepcopy(by_student),
        "by_school": deepcopy(by_school),
    }

def run_boston(students, schools, capacities, preferences, priorities):
    """Immediate-acceptance Boston mechanism with explicit round history."""
    by_student = {student: None for student in students}
    by_school = {school: [] for school in schools}
    history = [snapshot(0, "Start", by_student, by_school)]
    max_rounds = max(len(preferences[student]) for student in students)
    for rank in range(max_rounds):
        applications = {school: [] for school in schools}
        for student in students:
            if by_student[student] is None and rank < len(preferences[student]):
                applications[preferences[student][rank]].append(student)
        for school in schools:
            vacancies = capacities[school] - len(by_school[school])
            ordered = sorted(applications[school], key=lambda s: priority_rank(priorities, school, s))
            for student in ordered[:max(0, vacancies)]:
                by_student[student] = school
                by_school[school].append(student)
        history.append(snapshot(rank + 1, f"Round {rank + 1}: acceptances are final", by_student, by_school, applications))
    return {"by_student": by_student, "by_school": by_school, "history": history}

def run_deferred_acceptance(students, schools, capacities, preferences, priorities):
    """Student-proposing deferred acceptance with explicit round history."""
    by_student = {student: None for student in students}
    by_school = {school: [] for school in schools}
    next_choice = {student: 0 for student in students}
    history = [snapshot(0, "Start", by_student, by_school)]
    round_number = 0
    while any(by_student[s] is None and next_choice[s] < len(preferences[s]) for s in students):
        round_number += 1
        applications = {school: [] for school in schools}
        for student in students:
            if by_student[student] is None and next_choice[student] < len(preferences[student]):
                school = preferences[student][next_choice[student]]
                next_choice[student] += 1
                applications[school].append(student)
        for school in schools:
            pool = by_school[school] + applications[school]
            ordered = sorted(pool, key=lambda s: priority_rank(priorities, school, s))
            held = ordered[:capacities[school]]
            for student in pool:
                by_student[student] = school if student in held else None
            by_school[school] = held
        history.append(snapshot(round_number, f"Round {round_number}: highest-priority applicants are held", by_student, by_school, applications))
    history.append(snapshot(round_number + 1, "Final: tentative holds become assignments", by_student, by_school))
    return {"by_student": by_student, "by_school": by_school, "history": history}

def blocking_pairs(result, students, schools, capacities, preferences, priorities):
    """Return student-school pairs that block the displayed assignment."""
    pairs = []
    for student in students:
        current = result["by_student"][student]
        for school in schools:
            if school == current:
                continue
            prefers_school = current is None or preferences[student].index(school) < preferences[student].index(current)
            assigned = result["by_school"][school]
            school_would_accept = (
                len(assigned) < capacities[school]
                or any(priority_rank(priorities, school, student) < priority_rank(priorities, school, other) for other in assigned)
            )
            if prefers_school and school_would_accept:
                pairs.append((student, school))
    return pairs

def allocation_frame(result, label):
    return pd.DataFrame({
        "Student": students if (students := list(result["by_student"])) else [],
        label: [result["by_student"][student] or "Unmatched" for student in students],
    })

In [3]:
boston = run_boston(STUDENTS, SCHOOLS, CAPACITIES, PREFERENCES, PRIORITIES)
da = run_deferred_acceptance(STUDENTS, SCHOOLS, CAPACITIES, PREFERENCES, PRIORITIES)

comparison = allocation_frame(boston, "Boston").merge(allocation_frame(da, "Deferred acceptance"), on="Student")
comparison["Boston blocking pair?"] = ["Bo + Aurora" if student == "Bo" else "" for student in comparison["Student"]]
display(comparison)
print("Boston blocking pairs:", blocking_pairs(boston, STUDENTS, SCHOOLS, CAPACITIES, PREFERENCES, PRIORITIES))
print("DA blocking pairs:", blocking_pairs(da, STUDENTS, SCHOOLS, CAPACITIES, PREFERENCES, PRIORITIES))

assert boston["by_student"]["Bo"] == "Cedar"
assert da["by_student"]["Bo"] == "Aurora"
assert blocking_pairs(boston, STUDENTS, SCHOOLS, CAPACITIES, PREFERENCES, PRIORITIES) == [("Bo", "Aurora")]
assert blocking_pairs(da, STUDENTS, SCHOOLS, CAPACITIES, PREFERENCES, PRIORITIES) == []

,Student,Boston,Deferred acceptance,Boston blocking pair?
0,Amina,Beacon,Beacon,
1,Bo,Cedar,Aurora,Bo + Aurora
2,Chen,Aurora,Cedar,
3,Dara,Cedar,Cedar,


Boston blocking pairs: [('Bo', 'Aurora')]
DA blocking pairs: []


### Interpret the result in three voices

- **Game theorist:** Under Boston, Bo's truthful report leads to Cedar. Is there another report that improves Bo's assignment while others' reports stay fixed?
- **Social-choice researcher:** The Boston allocation violates stability in this instance because Bo prefers Aurora to Cedar and Aurora gives higher priority to Bo than to Chen. Which social criteria should be decisive, and why?
- **Mechanism designer:** Student-proposing deferred acceptance changes provisional-hold rules so the later proposal from Bo can displace Chen at Aurora. The observed instance is stable. The general strategy-proofness and stability claims require the theorem's standard assumptions; the simulation illustrates rather than proves them.

In [4]:
def animate_history(result, title):
    'Return a compact, accessible CSS animation of the computed round history.'
    slug = "matching-" + "-".join(title.lower().replace(":", "").split())
    frames = []
    for index, step in enumerate(result["history"]):
        applications = []
        for school in SCHOOLS:
            applicants = step.get("applications", {}).get(school, [])
            if applicants:
                applications.append(f"<span><b>{escape(school)}</b> &#8592; {escape(', '.join(applicants))}</span>")
        if not applications:
            applications.append("<span>No applications yet</span>")
        assignments = "".join(
            f"<div><b>{escape(student)}</b><span>&#8594; {escape(school or 'unassigned')}</span></div>"
            for student, school in step["by_student"].items()
        )
        frames.append(
            f'''<section class="matching-frame" aria-hidden="{'false' if index == 0 else 'true'}">
                  <p class="round">{escape(step['label'])}</p>
                  <div class="applications">{''.join(applications)}</div>
                  <div class="assignments">{assignments}</div>
                </section>'''
        )

    interval = 1.6
    duration = interval * len(frames)
    visible = max(8, 100 / len(frames) - 3)
    delays = "".join(
        f"#{slug} .matching-frame:nth-child({index + 1}){{animation-delay:-{index * interval:.1f}s}}"
        for index in range(len(frames))
    )
    return HTML(f'''<style>
      #{slug}{{background:#071326;color:#edf7ff;border:1px solid #214364;border-radius:16px;padding:18px;font-family:Arial,sans-serif;min-height:250px;overflow:hidden}}
      #{slug} .stage{{display:grid}}
      #{slug} .matching-frame{{grid-area:1/1;opacity:0;animation:{slug}-cycle {duration:.1f}s infinite}}
      #{slug} .round{{color:#ffcc66;font-weight:700;font-size:18px;margin:0 0 12px}}
      #{slug} .applications{{display:flex;gap:8px;flex-wrap:wrap;margin-bottom:14px}}
      #{slug} .applications span{{background:#241f46;border:1px dashed #9b7cff;border-radius:999px;padding:6px 10px}}
      #{slug} .assignments{{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:8px}}
      #{slug} .assignments div{{display:flex;justify-content:space-between;gap:12px;background:#10263f;border-left:3px solid #35d4ff;border-radius:8px;padding:10px}}
      {delays}
      @keyframes {slug}-cycle{{0%,{visible:.1f}%{{opacity:1}}{visible + 2:.1f}%,100%{{opacity:0}}}}
      @media (prefers-reduced-motion:reduce){{#{slug} .matching-frame{{display:none;animation:none}}#{slug} .matching-frame:first-child{{display:block;opacity:1}}}}
    </style>
    <div id="{slug}" class="matching-animation" role="img" aria-label="{escape(title)}: animated matching rounds">
      <p><b>{escape(title)}</b> · computed applications and current assignments</p>
      <div class="stage">{''.join(frames)}</div>
    </div>''')

display(animate_history(boston, "Boston mechanism: acceptances are final"))
display(animate_history(da, "Deferred acceptance: schools hold the best applicants tentatively"))

In [5]:
# A falsification test: can Bo benefit under Boston by changing only Bo's report?
strategic_preferences = deepcopy(PREFERENCES)
strategic_preferences["Bo"] = ["Aurora", "Beacon", "Cedar"]
strategic_boston = run_boston(STUDENTS, SCHOOLS, CAPACITIES, strategic_preferences, PRIORITIES)

true_rank = {school: rank for rank, school in enumerate(PREFERENCES["Bo"])}
truthful_assignment = boston["by_student"]["Bo"]
strategic_assignment = strategic_boston["by_student"]["Bo"]
print(f"Truthful Boston assignment for Bo: {truthful_assignment}")
print(f"Boston assignment after Bo reports Aurora first: {strategic_assignment}")
print(f"Under Bo's true preferences, {strategic_assignment} ranks above {truthful_assignment}:", true_rank[strategic_assignment] < true_rank[truthful_assignment])
assert true_rank[strategic_assignment] < true_rank[truthful_assignment]

Truthful Boston assignment for Bo: Cedar
Boston assignment after Bo reports Aurora first: Aurora
Under Bo's true preferences, Aurora ranks above Cedar: True


## Your change-one-input investigation

Change **one** input—one student's preferences, one school's priority, or one capacity—and complete this record before changing anything else:

1. **Prediction:** Which applications, tentative/final decisions, and allocation will change?
2. **Game-theory interpretation:** Which player's strategy or incentive changed?
3. **Social-choice interpretation:** Which collective criterion improved or worsened?
4. **Mechanism-design interpretation:** Did the stability or manipulation counterexample survive? Why?
5. **Verification:** Run both algorithms, inspect every round, and compare the computed result with your prediction.
6. **Boundary:** What does this single simulation fail to establish?

### Human-led AI protocol

First solve and record your reasoning. Then use a **bounded** AI prompt to request one counterexample or edge case. Verify every mathematical and bibliographic claim against the code, the cited paper, or an authoritative source. Revise your explanation and disclose what AI suggested, what you rejected, and why.

## Transfer to the final project

No matter your research topic, your future-research roadmap should include:

- **Joint perspective:** show how game theory, social choice, and mechanism design ask different but connected questions about the same problem.
- **Application transfer:** discuss what your model reveals when interpreted as an auction/allocation problem and as a voting/matching/collective-decision problem. State where the analogy breaks.
- **Computational artifact:** document pseudocode, executable code, simulation results, assumptions, edge cases, and reproducibility instructions in GitHub.
- **Economic and behavioral account:** explain incentives and welfare, then identify where actual behavior may depart from the formal model and what evidence would distinguish explanations.
- **Interdisciplinary frontier:** identify an intellectual contribution and practical impact across computer science, economics, and behavioral science; state open questions and who could benefit or bear risk.

Your notebook output is evidence only when a reader can reproduce it, inspect the mechanism, and understand what would falsify your claim.